In [1]:
import geopandas as gpd
import numpy as np
import pandas as pd

In [2]:
res_gdf = gpd.read_file('../clean_summarize/out_polys/sentinel_2021_v7_aea_cleaned_no0.gpkg')
res_gdf['geometry'] = res_gdf.make_valid()
res_gdf['area_ha'] = res_gdf.geometry.area/10000
res_gdf['source'] = 'resid'
res_gdf = res_gdf.loc[res_gdf['area_ha']>=0.05]
res_gdf = res_gdf.loc[res_gdf['area_ha']<=50]

In [3]:
ana_gdf = gpd.read_file('../compare_previous_results/data/ana/Massas_d_Agua.shp')
ana_gdf = ana_gdf.loc[ana_gdf['detipomass']=='Artificial']
ana_gdf = ana_gdf.to_crs('ESRI:102033')
ana_gdf['area_ha'] = ana_gdf.geometry.area/10000
ana_gdf['source'] = 'ana'
ana_gdf = ana_gdf.loc[ana_gdf['area_ha']>=0.05]
ana_gdf = ana_gdf.loc[ana_gdf['area_ha']<=50]

In [4]:
gdw_gdf = gpd.read_file('../../../../reservoir-id-cnn/reservoir-id-cnn/analysis/other_dams/gdw/GDW_v1_0_shp/GDW_reservoirs_v1_0.shp')
gdw_gdf = gdw_gdf.loc[gdw_gdf['COUNTRY']=='Brazil']
gdw_gdf = gdw_gdf.to_crs('ESRI:102033')
gdw_gdf['area_ha'] = gdw_gdf.geometry.area/10000
gdw_gdf['source'] = 'gdw'
gdw_gdf = gdw_gdf.loc[gdw_gdf['area_ha']>=0.05]
gdw_gdf = gdw_gdf.loc[gdw_gdf['area_ha']<=50]

In [5]:

car_gdf = gpd.read_file('../compare_previous_results/data/car/full_reservoirs_dissolve_explode.shp')
car_gdf = car_gdf.to_crs('ESRI:102033')
car_gdf['geometry'] = car_gdf.make_valid()
car_gdf['area_ha'] = car_gdf.geometry.area/10000
car_gdf['source'] = 'car'
car_gdf = car_gdf.loc[car_gdf.is_valid]
car_gdf = car_gdf.loc[car_gdf['area_ha']>=0.05]
car_gdf = car_gdf.loc[car_gdf['area_ha']<=50]

/home/ksolvik/miniconda3/envs/resgis/lib/python3.10/site-packages/shapely/constructive.py:532: RuntimeWarning: invalid value encountered in make_valid
  return lib.make_valid(geometry, **kwargs)


In [6]:
mb_gdf = gpd.read_file('../compare_previous_results/data/mapbiomas/v3_aea.gpkg')
mb_gdf = mb_gdf.loc[mb_gdf['DN']==2]
mb_gdf = mb_gdf.to_crs('ESRI:102033')
mb_gdf['geometry'] = mb_gdf.make_valid()
mb_gdf['area_ha'] = mb_gdf.geometry.area/10000
mb_gdf['source'] = 'mb'
mb_gdf = mb_gdf.loc[mb_gdf.is_valid]
mb_gdf = mb_gdf.loc[mb_gdf['area_ha']>=0.05]
mb_gdf = mb_gdf.loc[mb_gdf['area_ha']<=50]

In [7]:
# For each compared to reservoir ID: 
# 1. Total number and area intersection
# 2. Total number and area of union
def intersection_union(df1, df2):
    df1['id1'] = np.arange(df1.shape[0])
    df2['id2'] = np.arange(df2.shape[0])
    df_intersect = df1.sjoin(df2, how='inner')
    df_intersect['area_max'] = df_intersect[['area_ha_left', 'area_ha_right']].max(axis=1)
    df1_unique = df1.loc[~df1['id1'].isin(df_intersect['id1'])]
    df2_unique = df2.loc[~df2['id2'].isin(df_intersect['id2'])]
    out_dict = {
        'name': df1['source'].values[0] + '-' + df2['source'].values[0],
        'left_area': df1['area_ha'].sum(),
        'left_count': df1.shape[0],
        'right_area': df2['area_ha'].sum(),
        'right_count': df2.shape[0],
        'count_intersect': df_intersect.shape[0],
        'left_area_intersect': df_intersect['area_ha_left'].sum(),
        'right_area_intersect': df_intersect['area_ha_right'].sum(),
        'union_count': df_intersect.shape[0] + df1_unique.shape[0] + df2_unique.shape[0],
        'union_area': df_intersect['area_max'].sum() + df1_unique['area_ha'].sum() + df2_unique['area_ha'].sum(),
        'left_area_union': df_intersect['area_ha_left'].sum() + df1_unique['area_ha'].sum() + df2_unique['area_ha'].sum(),
        'right_area_union': df_intersect['area_ha_right'].sum() + df2_unique['area_ha'].sum() + df1_unique['area_ha'].sum(),
    }
    return out_dict


In [8]:
out_dicts = []
out_dicts.append(intersection_union(res_gdf, gdw_gdf))
out_dicts.append(intersection_union(res_gdf, ana_gdf))
out_dicts.append(intersection_union(res_gdf, car_gdf))
out_dicts.append(intersection_union(res_gdf, mb_gdf))

In [9]:
pd.DataFrame(out_dicts)

,name,left_area,left_count,right_area,right_count,count_intersect,left_area_intersect,right_area_intersect,union_count,union_area,left_area_union,right_area_union
0,resid-gdw,763187.11,1131418,70944.293464,3823,3968,45354.93,75248.384069,1131983,8.107128e+05,7.762242e+05,8.061177e+05
1,resid-ana,763187.11,1131418,529908.237967,171665,137797,345081.31,487886.145536,1175283,1.104822e+06,9.035461e+05,1.046351e+06
2,resid-car,763187.11,1131418,560228.512891,502936,248397,325237.06,462056.813881,1412509,1.217663e+06,9.941613e+05,1.130981e+06
3,resid-mb,763187.11,1131418,531552.815140,688360,438975,785709.30,403392.333199,1453335,1.239069e+06,1.175114e+06,7.927969e+05


In [10]:
# Minimum check: *One* of the other datasets must agree
all_indices = []
for compare in [gdw_gdf, car_gdf, mb_gdf, ana_gdf]:
    all_indices.append(res_gdf.sjoin(compare).index.values)
res_minimum = res_gdf.loc[np.unique(np.concatenate(all_indices))]


In [11]:

res_minimum = res_gdf.loc[np.unique(np.concatenate(all_indices))]
print(res_minimum.shape)
print(res_minimum['area_ha'].sum())

(507938, 5)
611662.99


In [12]:
def quick_union(df1, df2):
    df1 = df1[['area_ha','geometry']].copy()
    df2 = df2[['area_ha','geometry']].copy()
    df1['id1'] = np.arange(df1.shape[0])
    df2['id2'] = np.arange(df2.shape[0])
    df_intersect = df1.sjoin(df2, how='inner').fillna(0)
    df_intersect['area_ha'] = df_intersect[['area_ha_left', 'area_ha_right']].max(axis=1)
    df1_unique = df1.loc[~df1['id1'].isin(df_intersect['id1'])]
    df2_unique = df2.loc[~df2['id2'].isin(df_intersect['id2'])]
    df_union = pd.concat([df_intersect, df1_unique, df2_unique], axis=0, ignore_index=True)
    return df_union.drop(columns=['index_right', 'id2','id1', 'area_ha_left','area_ha_right']).reset_index().drop(columns=['index'])




In [13]:
# Absolute maximum: get union of all datasets, taking maximum area of each
temp_gdf = quick_union(res_gdf, gdw_gdf)
temp_gdf = quick_union(temp_gdf, ana_gdf)
temp_gdf = quick_union(temp_gdf, mb_gdf)
temp_gdf = quick_union(temp_gdf, car_gdf)

In [14]:
temp_gdf.shape

(1770646, 2)

In [15]:
temp_gdf['area_ha'].sum()

2279533.014105323